# Level 1 — Task 1: Data Cleaning and Preprocessing
**Codveda Technology | Data Analytics Internship**

> **Dataset:** `churnbigml80.csv` | **Tools:** Python, pandas
>
> **Objective:** Load a raw customer churn dataset, identify and handle missing values,
> remove duplicates, standardize inconsistent formats, and engineer useful new features.


## Step 1 — Imports and Setup

In [3]:
import pandas as pd
import numpy as np
import os

os.makedirs("outputs/level1", exist_ok=True)
print("Libraries imported. Output folder ready.")

Libraries imported. Output folder ready.


## Step 2 — Load the Dataset

In [4]:
df = pd.read_csv("churn-bigml-80.csv")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Shape: 2666 rows × 20 columns


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


## Step 3 — Initial Audit (dtypes, nulls, statistics)

In [5]:
print("Data types:")
print(df.dtypes)
print()
print("Descriptive statistics:")
df.describe().round(2)

Data types:
State                      object
Account length              int64
Area code                   int64
International plan         object
Voice mail plan            object
Number vmail messages       int64
Total day minutes         float64
Total day calls             int64
Total day charge          float64
Total eve minutes         float64
Total eve calls             int64
Total eve charge          float64
Total night minutes       float64
Total night calls           int64
Total night charge        float64
Total intl minutes        float64
Total intl calls            int64
Total intl charge         float64
Customer service calls      int64
Churn                        bool
dtype: object

Descriptive statistics:


,Account length,Area code,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls
count,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00,2666.00
mean,100.62,437.44,8.02,179.48,100.31,30.51,200.39,100.02,17.03,201.17,100.11,9.05,10.24,4.47,2.76,1.56
std,39.56,42.52,13.61,54.21,19.99,9.22,50.95,20.16,4.33,50.78,19.42,2.29,2.79,2.46,0.75,1.31
min,1.00,408.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,43.70,33.00,1.97,0.00,0.00,0.00,0.00
25%,73.00,408.00,0.00,143.40,87.00,24.38,165.30,87.00,14.05,166.92,87.00,7.51,8.50,3.00,2.30,1.00
50%,100.00,415.00,0.00,179.95,101.00,30.59,200.90,100.00,17.08,201.15,100.00,9.05,10.20,4.00,2.75,1.00
75%,127.00,510.00,19.00,215.90,114.00,36.70,235.10,114.00,19.98,236.48,113.00,10.64,12.10,6.00,3.27,2.00
max,243.00,510.00,50.00,350.80,160.00,59.64,363.70,170.00,30.91,395.00,166.00,17.77,20.00,20.00,5.40,9.00


In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print("Missing values per column:")
print(audit[audit["Missing Count"] > 0] if audit["Missing Count"].sum() > 0 else "None found ✓")

Missing values per column:
None found ✓


## Step 4 — Handle Missing Values

> Inject synthetic NaNs for demonstration, then impute numeric columns with **median** and categorical columns with **mode**.

In [7]:
# Inject synthetic NaNs to demonstrate imputation
df.loc[df.sample(frac=0.02, random_state=42).index, "Total day minutes"] = np.nan
df.loc[df.sample(frac=0.01, random_state=7).index, "Customer service calls"] = np.nan
print("NaNs introduced:", df[["Total day minutes","Customer service calls"]].isnull().sum().to_dict())

# Impute
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(include="object").columns

for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after imputation:", df.isnull().sum().sum(), "✓")

NaNs introduced: {'Total day minutes': 53, 'Customer service calls': 27}
Missing values after imputation: 0 ✓


C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_4124\3204336321.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_4124\3204336321.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

## Step 5 — Remove Duplicate Rows

In [8]:
n_before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {n_before - len(df)}  |  Rows remaining: {len(df)}")

Duplicates removed: 0  |  Rows remaining: 2666


## Step 6 — Standardize Data Formats

In [9]:
# Strip whitespace
for col in cat_cols:
    df[col] = df[col].str.strip()

# Encode Yes/No columns as integers
for col in ["International plan", "Voice mail plan"]:
    df[col] = (df[col] == "Yes").astype(int)

# Churn: True/False → 1/0
df["Churn"] = df["Churn"].astype(int)

# State: uppercase
df["State"] = df["State"].str.upper()

print("Standardization complete.")
df[["State","International plan","Voice mail plan","Churn"]].head()

Standardization complete.


,State,International plan,Voice mail plan,Churn
0,KS,0,1,0
1,OH,0,1,0
2,NJ,0,0,0
3,OH,1,0,0
4,OK,1,0,0


## Step 7 — Feature Engineering

In [10]:
df["Total minutes"] = (df["Total day minutes"] + df["Total eve minutes"]
                       + df["Total night minutes"] + df["Total intl minutes"])
df["Total charges"] = (df["Total day charge"] + df["Total eve charge"]
                       + df["Total night charge"] + df["Total intl charge"])

print("New features created: 'Total minutes', 'Total charges'")
df[["Total minutes","Total charges"]].describe().round(2)

New features created: 'Total minutes', 'Total charges'


,Total minutes,Total charges
count,2666.00,2666.00
mean,591.15,59.36
std,89.83,10.48
min,284.30,22.93
25%,530.33,52.29
50%,593.85,59.36
75%,653.00,66.57
max,885.00,96.15


## Step 8 — Final Validation and Save

In [11]:
print(f"Final shape     : {df.shape}")
print(f"Missing values  : {df.isnull().sum().sum()}")
print(f"Duplicate rows  : {df.duplicated().sum()}")

output_path = "outputs/level1/churn_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved → {output_path}")
df.head()

Final shape     : (2666, 22)
Missing values  : 0
Duplicate rows  : 0

Cleaned dataset saved → outputs/level1/churn_cleaned.csv


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,Total minutes,Total charges
0,KS,128,415,0,1,25,265.1,110,45.07,197.4,...,244.7,91,11.01,10.0,3,2.70,1.0,0,717.2,75.56
1,OH,107,415,0,1,26,161.6,123,27.47,195.5,...,254.4,103,11.45,13.7,3,3.70,1.0,0,625.2,59.24
2,NJ,137,415,0,0,0,243.4,114,41.38,121.2,...,162.6,104,7.32,12.2,5,3.29,0.0,0,539.4,62.29
3,OH,84,408,1,0,0,299.4,71,50.90,61.9,...,196.9,89,8.86,6.6,7,1.78,2.0,0,564.8,66.80
4,OK,75,415,1,0,0,166.7,113,28.34,148.3,...,186.9,121,8.41,10.1,3,2.73,3.0,0,512.0,52.09
